# Day 4 · Exercise 3: Pydantic Validation

**What you'll build:** `validate_person` — validate a raw dict against a Pydantic schema.

**Why it matters:** Pydantic is a general validation tool, completely independent of LLMs. It enforces types, catches missing fields, and raises a clear `ValidationError` at the point of entry — not somewhere deep in your pipeline. This exercise teaches that layer before combining it with the others.

> **No Ollama needed** for this exercise — it's pure Python.

## The Schema (already defined)

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class PersonProfile(BaseModel):
    name:   str       = Field(description="Full name of the person")
    age:    int       = Field(description="Age in years")
    city:   str       = Field(description="City where they live")
    skills: list[str] = Field(description="List of professional skills")
    bio:    str       = Field(description="One-sentence biography")

## Your Implementation

In [ ]:
def validate_person(data: dict) -> PersonProfile:
    """Validate a raw dict against the PersonProfile schema.

    Call PersonProfile.model_validate(data) and return the result.
    Pydantic will raise ValidationError automatically if the data
    doesn't match the schema.

    Args:
        data: A dict with keys name, age, city, skills, bio.

    Returns:
        A validated PersonProfile instance.

    Raises:
        ValidationError: if the data doesn't match the schema.

    Example:
        profile = validate_person({
            "name": "Alice", "age": 30, "city": "Cape Town",
            "skills": ["Python"], "bio": "Data engineer."
        })
        profile.age  # 30 (integer, not string)
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

VALID_DATA = {
    "name": "Alice Chen",
    "age": 30,
    "city": "Cape Town",
    "skills": ["Python", "SQL"],
    "bio": "A data engineer with 5 years of experience.",
}

def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and is callable
    try:
        assert callable(validate_person), 'validate_person is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: returns a PersonProfile for valid data
    profile = None
    try:
        profile = validate_person(VALID_DATA)
        assert isinstance(profile, PersonProfile), \
            f'expected PersonProfile, got {type(profile).__name__}'
        print(f'{_PASS} Check 2/{total}: returns a PersonProfile instance for valid data')
        score += 1
    except ValidationError as e:
        print(f'{_FAIL} Check 2/{total}: ValidationError on valid data — {e}')
    except AssertionError as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: unexpected error — {e}')

    # Check 3: field values are correctly typed
    if profile is not None:
        try:
            assert isinstance(profile.name, str), f'name should be str, got {type(profile.name).__name__}'
            assert isinstance(profile.age, int), f'age should be int, got {type(profile.age).__name__}'
            assert isinstance(profile.skills, list), f'skills should be list, got {type(profile.skills).__name__}'
            assert profile.name == "Alice Chen", f'name mismatch: {profile.name!r}'
            assert profile.age == 30, f'age mismatch: {profile.age}'
            print(f'{_PASS} Check 3/{total}: field types are correct (name: str, age: int, skills: list)')
            score += 1
        except AssertionError as e:
            print(f'{_FAIL} Check 3/{total}: {e}')
    else:
        print(f'{_FAIL} Check 3/{total}: skipped (no profile from check 2)')

    # Check 4: ValidationError raised for invalid data
    try:
        validate_person({"name": "Eve", "age": "not-a-number",
                         "city": "Durban", "skills": [], "bio": "..."})
        print(f'{_FAIL} Check 4/{total}: should have raised ValidationError for non-numeric age')
    except ValidationError:
        print(f'{_PASS} Check 4/{total}: ValidationError raised for invalid data (age="not-a-number")')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: unexpected error — {e}')

    print()
    if score == total:
        print('=' * 52)
        print(f'  {_PASS}  Exercise 3 complete! {total}/{total} checks passed.')
        print('=' * 52)
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

Explore what Pydantic's `ValidationError` tells you:

```python
try:
    validate_person({
        "name": "Bob",
        # age missing entirely
        "city": "Pretoria",
        "skills": "Python",  # wrong type — should be list
        "bio": 42,           # wrong type — should be str
    })
except ValidationError as e:
    print(e)  # notice: multiple errors reported at once
    print()
    for err in e.errors():
        print(f"Field '{err['loc'][0]}': {err['msg']}")
```

Pydantic collects ALL errors before raising, so you see every problem at once — not just the first one.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
def validate_person(data: dict) -> PersonProfile:
    return PersonProfile.model_validate(data)
```

**Why this works:** `model_validate` reads the data, checks each field against the type annotation, coerces where possible (e.g. an integer `"30"` might be coerced to `int` 30 — but `"thirty"` cannot be coerced and raises `ValidationError`), and returns a fully typed `PersonProfile` instance. The function body is one line because Pydantic handles all the complexity. Note: use `model_validate` (v2 API), not `parse_obj` (v1 — deprecated).
</details>